In [1]:
# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================

!pip install -q transformers datasets evaluate textstat \
sacremoses sacrebleu rouge_score bert_score accelerate

import os
import gc
import time
import re

import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import huggingface_hub
import evaluate
import textstat

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

print("=" * 100)
print("🔍 ENVIRONMENT CHECK")
print("=" * 100)

print("PyTorch         :", torch.__version__)
print("Transformers    :", transformers.__version__)
print("Datasets        :", datasets.__version__)
print("HuggingFace Hub :", huggingface_hub.__version__)
print("Evaluate        :", evaluate.__version__)

print("\nCUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    total_vram = (
        torch.cuda.get_device_properties(0)
        .total_memory
        / (1024 ** 3)
    )

    print(
        "Total VRAM:",
        round(total_vram, 2),
        "GB"
    )

print("=" * 100)

🔍 ENVIRONMENT CHECK
PyTorch         : 2.11.0+cu128
Transformers    : 5.15.0
Datasets        : 4.0.0
HuggingFace Hub : 1.27.0
Evaluate        : 0.4.6

CUDA Available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


In [2]:
# ============================================================
# 📂 LOAD ASSET — VALIDATION + TEST
# ============================================================

print("=" * 100)
print("📂 LOADING ASSET — VALIDATION + TEST")
print("=" * 100)

asset = load_dataset(
    "facebook/asset"
)

print("\nDataset loaded:")
print(asset)


# ============================================================
# COMBINE VALIDATION + TEST
# ============================================================

asset_df = pd.concat(
    [
        asset["validation"].to_pandas(),
        asset["test"].to_pandas()
    ],
    ignore_index=True
)


# ============================================================
# VERIFICATION
# ============================================================

validation_count = len(
    asset["validation"]
)

test_count = len(
    asset["test"]
)

total_count = len(
    asset_df
)

total_references = (
    asset_df["simplifications"]
    .apply(len)
    .sum()
)


print("\n" + "=" * 100)
print("🔍 ASSET VERIFICATION")
print("=" * 100)

print(
    f"Validation samples: {validation_count}"
)

print(
    f"Test samples      : {test_count}"
)

print(
    f"Total samples     : {total_count}"
)

print(
    f"Total references  : {total_references}"
)


# ============================================================
# STRICT CHECKS
# ============================================================

assert validation_count == 2000

assert test_count == 359

assert total_count == 2359

assert (
    asset_df["simplifications"]
    .apply(len)
    .eq(10)
    .all()
)

assert total_references == 23590


print("\n✅ ASSET verification passed.")

print(
    "\nFirst sample:"
)

print(
    asset_df.iloc[0]
)

print("\n" + "=" * 100)
print("✅ ASSET DATASET READY")
print("=" * 100)

📂 LOADING ASSET — VALIDATION + TEST



Dataset loaded:
DatasetDict({
    validation: Dataset({
        features: ['original', 'simplifications'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['original', 'simplifications'],
        num_rows: 359
    })
})

🔍 ASSET VERIFICATION
Validation samples: 2000
Test samples      : 359
Total samples     : 2359
Total references  : 23590

✅ ASSET verification passed.

First sample:
original           Adjacent counties are Marin (to the south), Me...
simplifications    [countries next to it are Marin, Mendocino, La...
Name: 0, dtype: object

✅ ASSET DATASET READY


In [3]:
# ============================================================
# 🔥 QWEN2.5-3B-INSTRUCT — LOAD MODEL
# ============================================================

MODEL_NAME = (
    "Qwen/Qwen2.5-3B-Instruct"
)

MAX_NEW_TOKENS = 128

CHECKPOINT_EVERY = 25


print("=" * 100)
print("🔥 QWEN2.5-3B-INSTRUCT — CoT EVALUATION")
print("=" * 100)

print(
    "Model:",
    MODEL_NAME
)

print(
    "CUDA:",
    torch.cuda.is_available()
)


# ============================================================
# LOAD TOKENIZER
# ============================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

if tokenizer.pad_token is None:

    tokenizer.pad_token = (
        tokenizer.eos_token
    )

print("✅ Tokenizer loaded.")


# ============================================================
# LOAD MODEL
# ============================================================

print("\nLoading Qwen2.5-3B-Instruct...")

model = AutoModelForCausalLM.from_pretrained(

    MODEL_NAME,

    torch_dtype=torch.float16,

    device_map="auto"
)

model.eval()


print("\n✅ Qwen loaded successfully.")

print(
    "Model class:",
    model.__class__.__name__
)

print("=" * 100)

🔥 QWEN2.5-3B-INSTRUCT — CoT EVALUATION
Model: Qwen/Qwen2.5-3B-Instruct
CUDA: True

Loading tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ Qwen loaded successfully.
Model class: Qwen2ForCausalLM


In [4]:
# ============================================================
# 🧠 QWEN — CHAIN-OF-THOUGHT PROMPT
# ============================================================

def cot_prompt(sentence):

    return f"""You are a text simplification system.

Simplify the original sentence into ONE clear and easy-to-understand sentence.

Before answering, silently analyze:
1. The main meaning of the sentence.
2. Important facts and relationships that must be preserved.
3. Difficult words or phrases that can be replaced with simpler language.
4. How to make the sentence easier to understand without changing its meaning.

Preserve the original meaning.
Do not add information.
Do not remove important information.

IMPORTANT:
Do not show your reasoning.
Do not provide explanations.
Do not ask questions.
Do not discuss your answer.
Do not continue after the simplified sentence.

Return ONLY the final simplified sentence.

Original sentence:
{sentence}

Simplified sentence:"""


print("✅ Exact Qwen CoT prompt ready.")

✅ Exact Qwen CoT prompt ready.


In [5]:
# ============================================================
# 📊 LOAD ALL EVALUATION METRICS
# ============================================================

print("=" * 100)
print("📊 LOADING EVALUATION METRICS")
print("=" * 100)


# SARI

sari_metric = evaluate.load(
    "sari"
)


# BLEU

bleu_metric = evaluate.load(
    "bleu"
)


# ROUGE

rouge_metric = evaluate.load(
    "rouge"
)


# BERTScore

bertscore_metric = evaluate.load(
    "bertscore"
)


print("\n✅ SARI loaded")
print("✅ BLEU loaded")
print("✅ ROUGE-L loaded")
print("✅ BERTScore loaded")
print("✅ FKGL ready")

print("\n🎯 ALL METRICS ARE READY.")

📊 LOADING EVALUATION METRICS

✅ SARI loaded
✅ BLEU loaded
✅ ROUGE-L loaded
✅ BERTScore loaded
✅ FKGL ready

🎯 ALL METRICS ARE READY.


In [6]:
# ============================================================
# 📊 QWEN — EVALUATION FUNCTION
# ============================================================

def evaluate_one_sample(
    source,
    prediction,
    references
):

    prediction = str(
        prediction
    ).strip()


    # ========================================================
    # EMPTY OUTPUT SAFETY
    # ========================================================

    if not prediction:

        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }


    # ========================================================
    # SARI — ALL 10 REFERENCES
    # ========================================================

    try:

        sari_result = sari_metric.compute(

            sources=[source],

            predictions=[prediction],

            references=[references]
        )

        sari = sari_result[
            "sari"
        ]

    except Exception as e:

        print(
            "⚠️ SARI error:",
            e
        )

        sari = 0.0


    # ========================================================
    # BLEU — ALL 10 REFERENCES
    # ========================================================

    try:

        bleu_result = bleu_metric.compute(

            predictions=[
                prediction
            ],

            references=[
                references
            ],

            smooth=True
        )

        bleu = bleu_result[
            "bleu"
        ]

    except Exception as e:

        print(
            "⚠️ BLEU error:",
            e
        )

        bleu = 0.0


    # ========================================================
    # ROUGE-L — MEAN OF 10 REFERENCES
    # ========================================================

    try:

        rouge_scores = []

        for reference in references:

            rouge_result = (
                rouge_metric.compute(

                    predictions=[
                        prediction
                    ],

                    references=[
                        reference
                    ]
                )
            )

            rouge_scores.append(

                rouge_result[
                    "rougeL"
                ]
            )


        rouge_l = float(

            np.mean(
                rouge_scores
            )
        )

    except Exception as e:

        print(
            "⚠️ ROUGE-L error:",
            e
        )

        rouge_l = 0.0


    # ========================================================
    # BERTSCORE-F1 — MEAN OF 10 REFERENCES
    # ========================================================

    try:

        bert_result = (
            bertscore_metric.compute(

                predictions=[
                    prediction
                ] * len(references),

                references=references,

                lang="en"
            )
        )


        bertscore_f1 = float(

            np.mean(

                bert_result[
                    "f1"
                ]
            )
        )


        if np.isnan(
            bertscore_f1
        ):

            bertscore_f1 = 0.0


    except Exception as e:

        print(
            "⚠️ BERTScore error:",
            e
        )

        bertscore_f1 = 0.0


    # ========================================================
    # FKGL — GENERATED SENTENCE ONLY
    # ========================================================

    try:

        fkgl = (
            textstat
            .flesch_kincaid_grade(
                prediction
            )
        )


        if np.isnan(fkgl):

            fkgl = 0.0


    except Exception as e:

        print(
            "⚠️ FKGL error:",
            e
        )

        fkgl = 0.0


    # ========================================================
    # RETURN
    # ========================================================

    return {

        "sari":
            float(sari),

        "bleu":
            float(bleu),

        "rougeL":
            float(rouge_l),

        "bertscore_f1":
            float(bertscore_f1),

        "fkgl":
            float(fkgl)
    }


print(
    "✅ evaluate_one_sample() ready."
)

✅ evaluate_one_sample() ready.


In [7]:
# ============================================================
# 🧹 QWEN OUTPUT CLEANER
# ============================================================

def clean_qwen_output(raw_output):

    if raw_output is None:

        return ""


    text = str(
        raw_output
    ).strip()


    if not text:

        return ""


    # ========================================================
    # REMOVE COMMON LABELS
    # ========================================================

    labels = [

        "Simplified sentence:",
        "Simplified Sentence:",

        "Final simplified sentence:",
        "Final Simplified Sentence:",

        "Final answer:",
        "Final Answer:",

        "Answer:"
    ]


    for label in labels:

        if text.startswith(label):

            text = text[
                len(label):
            ].strip()


    # ========================================================
    # REMOVE UNWANTED CONTINUATION
    # ========================================================

    unwanted_markers = [

        "Original sentence:",
        "Original Sentence:",

        "Reasoning:",
        "Explanation:",

        "The original sentence",
        "The simplified sentence",

        "Here is an explanation",
        "For example:",

        "Alternative:"
    ]


    cut_positions = []


    for marker in unwanted_markers:

        position = text.find(
            marker
        )

        if position > 0:

            cut_positions.append(
                position
            )


    if cut_positions:

        text = text[
            :min(cut_positions)
        ].strip()


    # ========================================================
    # REMOVE CODE BLOCKS
    # ========================================================

    text = text.replace(
        "```",
        ""
    )


    # ========================================================
    # REMOVE SURROUNDING QUOTES
    # ========================================================

    if (

        len(text) >= 2

        and text.startswith('"')

        and text.endswith('"')

    ):

        text = text[
            1:-1
        ].strip()


    return text.strip()


print(
    "✅ Qwen output cleaner ready."
)

✅ Qwen output cleaner ready.


In [8]:
# ============================================================
# 🧪 QWEN — SINGLE SAMPLE CoT TEST
# ============================================================

example = asset_df.iloc[0]


source = str(
    example["original"]
).strip()


references = [

    str(ref).strip()

    for ref in example[
        "simplifications"
    ]
]


# ============================================================
# CREATE PROMPT
# ============================================================

prompt = cot_prompt(
    source
)


# ============================================================
# QWEN CHAT TEMPLATE
# ============================================================

messages = [

    {
        "role": "user",
        "content": prompt
    }
]


formatted_prompt = (
    tokenizer.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True
    )
)


# ============================================================
# TOKENIZE
# ============================================================

inputs = tokenizer(

    formatted_prompt,

    return_tensors="pt",

    truncation=True,

    max_length=2048
)


inputs = {

    key: value.to(
        model.device
    )

    for key, value in inputs.items()
}


# ============================================================
# RESET VRAM
# ============================================================

if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats()


# ============================================================
# GENERATE
# ============================================================

start_time = time.perf_counter()


with torch.inference_mode():

    output_ids = model.generate(

        **inputs,

        max_new_tokens=MAX_NEW_TOKENS,

        do_sample=False,

        use_cache=True,

        pad_token_id=(
            tokenizer.pad_token_id
        ),

        eos_token_id=(
            tokenizer.eos_token_id
        )
    )


end_time = time.perf_counter()


# ============================================================
# DECODE ONLY GENERATED TOKENS
# ============================================================

generated_ids = output_ids[

    0,

    inputs["input_ids"].shape[1]:
]


raw_prediction = tokenizer.decode(

    generated_ids,

    skip_special_tokens=True

).strip()


prediction = clean_qwen_output(
    raw_prediction
)


# ============================================================
# PERFORMANCE
# ============================================================

inference_time = (
    end_time - start_time
)


if torch.cuda.is_available():

    peak_vram = (

        torch.cuda.max_memory_allocated()

        / (1024 ** 3)
    )

else:

    peak_vram = None


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("🧪 QWEN CoT — SINGLE SAMPLE TEST")
print("=" * 100)

print("\n🔵 ORIGINAL:")
print(source)

print("\n🟡 RAW OUTPUT:")
print(raw_prediction)

print("\n🟢 CLEANED OUTPUT:")
print(prediction)

print("\n⚡ PERFORMANCE")
print("-" * 60)

print(
    f"Inference Time: "
    f"{inference_time:.3f} sec"
)

if peak_vram is not None:

    print(
        f"Peak VRAM: "
        f"{peak_vram:.3f} GB"
    )


🧪 QWEN CoT — SINGLE SAMPLE TEST

🔵 ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

🟡 RAW OUTPUT:
Five counties border Marin: Mendocino to the north, Lake in the northeast, Napa to the east, and Solano and Contra Costa to the southeast.

🟢 CLEANED OUTPUT:
Five counties border Marin: Mendocino to the north, Lake in the northeast, Napa to the east, and Solano and Contra Costa to the southeast.

⚡ PERFORMANCE
------------------------------------------------------------
Inference Time: 6.095 sec
Peak VRAM: 5.781 GB


In [9]:
# ============================================================
# 🔥 QWEN2.5-3B-INSTRUCT — CHAIN-OF-THOUGHT
# ASSET — 2359 SAMPLES
# EVALUATION / INFERENCE ONLY
# ============================================================

MODEL_NAME = (
    "Qwen/Qwen2.5-3B-Instruct"
)

COT_OUTPUT = (
    "QWEN_COT_ASSET_2359_FINAL.csv"
)

MAX_NEW_TOKENS = 128

CHECKPOINT_EVERY = 25


print("=" * 100)
print("🔥 QWEN2.5-3B-INSTRUCT — CHAIN-OF-THOUGHT")
print("=" * 100)

print(
    f"Model: {MODEL_NAME}"
)

print(
    f"Samples: {len(asset_df)}"
)

print(
    "References per sample: 10"
)

print(
    f"Maximum new tokens: {MAX_NEW_TOKENS}"
)

print(
    f"Checkpoint every: {CHECKPOINT_EVERY}"
)

print("\nTraining: ❌ OFF")
print("Gradients: ❌ OFF")
print("Evaluation: ✅ ON")


# ============================================================
# VERIFY DATASET
# ============================================================

assert len(asset_df) == 2359

assert asset_df[
    "simplifications"
].apply(
    lambda x: len(x) == 10
).all()


print(
    "\n✅ ASSET dataset verification passed."
)


# ============================================================
# VERIFY MODEL
# ============================================================

assert tokenizer is not None

assert model is not None

model.eval()


print("✅ Tokenizer verified.")

print("✅ Qwen model verified.")


# ============================================================
# CHECKPOINT / RESUME
# ============================================================

print("\n" + "=" * 100)
print("📁 CHECKPOINT / RESUME SETUP")
print("=" * 100)


if os.path.exists(
    COT_OUTPUT
):

    cot_df = pd.read_csv(
        COT_OUTPUT
    )

    completed_ids = set(

        cot_df[
            "sample_id"
        ].astype(int)
    )

    print(
        f"🔄 Existing results: "
        f"{len(completed_ids)} completed."
    )

else:

    cot_df = pd.DataFrame()

    completed_ids = set()

    print(
        "🆕 No existing checkpoint."
    )

    print(
        "Starting from sample 1."
    )


# ============================================================
# START EVALUATION
# ============================================================

print("\n" + "=" * 100)

print(
    "🚀 STARTING QWEN CoT EVALUATION"
)

print("=" * 100)


for i in range(
    len(asset_df)
):

    # ========================================================
    # RESUME
    # ========================================================

    if i in completed_ids:

        continue


    # ========================================================
    # GET DATA
    # ========================================================

    example = asset_df.iloc[
        i
    ]


    source = str(
        example["original"]
    ).strip()


    references = [

        str(ref).strip()

        for ref in example[
            "simplifications"
        ]
    ]


    # ========================================================
    # CREATE CoT PROMPT
    # ========================================================

    prompt = cot_prompt(
        source
    )


    # ========================================================
    # QWEN CHAT TEMPLATE
    # ========================================================

    messages = [

        {
            "role": "user",
            "content": prompt
        }
    ]


    formatted_prompt = (
        tokenizer.apply_chat_template(

            messages,

            tokenize=False,

            add_generation_prompt=True
        )
    )


    # ========================================================
    # TOKENIZE
    # ========================================================

    inputs = tokenizer(

        formatted_prompt,

        return_tensors="pt",

        truncation=True,

        max_length=2048
    )


    inputs = {

        key: value.to(
            model.device
        )

        for key, value in inputs.items()
    }


    # ========================================================
    # RESET VRAM
    # ========================================================

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.reset_peak_memory_stats()


    # ========================================================
    # GENERATE
    # ========================================================

    raw_prediction = ""

    output_ids = None


    start_time = time.perf_counter()


    try:

        with torch.inference_mode():

            output_ids = model.generate(

                **inputs,

                max_new_tokens=MAX_NEW_TOKENS,

                do_sample=False,

                use_cache=True,

                pad_token_id=(

                    tokenizer.pad_token_id

                    if tokenizer.pad_token_id
                    is not None

                    else tokenizer.eos_token_id
                ),

                eos_token_id=(
                    tokenizer.eos_token_id
                )
            )


        # ====================================================
        # KEEP ONLY GENERATED TOKENS
        # ====================================================

        generated_ids = output_ids[

            0,

            inputs[
                "input_ids"
            ].shape[1]:

        ]


        raw_prediction = tokenizer.decode(

            generated_ids,

            skip_special_tokens=True

        ).strip()


    except Exception as e:

        print(

            f"\n❌ GENERATION ERROR "
            f"at sample {i + 1}: {e}"
        )

        raw_prediction = ""


    end_time = time.perf_counter()


    # ========================================================
    # INFERENCE TIME
    # ========================================================

    inference_time = (

        end_time - start_time
    )


    # ========================================================
    # CLEAN OUTPUT
    # ========================================================

    prediction = clean_qwen_output(
        raw_prediction
    )


    # ========================================================
    # EMPTY OUTPUT FALLBACK
    # ========================================================

    if not prediction:

        prediction = source


    # ========================================================
    # PEAK VRAM
    # ========================================================

    if torch.cuda.is_available():

        peak_vram = (

            torch.cuda.max_memory_allocated()

            / (1024 ** 3)
        )

    else:

        peak_vram = None


    # ========================================================
    # CALCULATE METRICS
    # ========================================================

    scores = evaluate_one_sample(

        source,

        prediction,

        references
    )


    # ========================================================
    # SAVE RESULT
    # ========================================================

    result = {

        "sample_id":
            i,

        "model":
            "Qwen2.5-3B-Instruct",

        "prompting":
            "chain-of-thought",

        "original":
            source,

        "generated":
            prediction,

        "raw_generated":
            raw_prediction,

        "references":
            " ||| ".join(
                references
            ),

        "sari":
            scores["sari"],

        "bleu":
            scores["bleu"],

        "rougeL":
            scores["rougeL"],

        "bertscore_f1":
            scores["bertscore_f1"],

        "fkgl":
            scores["fkgl"],

        "inference_time_sec":
            inference_time,

        "peak_vram_gb":
            peak_vram
    }


    # ========================================================
    # APPEND RESULT
    # ========================================================

    cot_df = pd.concat(

        [
            cot_df,

            pd.DataFrame(
                [result]
            )
        ],

        ignore_index=True
    )


    completed_ids.add(
        i
    )


    # ========================================================
    # CHECKPOINT
    # ========================================================

    if (

        len(completed_ids)
        % CHECKPOINT_EVERY == 0

        or

        i == len(asset_df) - 1
    ):

        cot_df.to_csv(

            COT_OUTPUT,

            index=False
        )


        print(

            f"\n💾 CHECKPOINT SAVED: "
            f"{len(completed_ids)}/"
            f"{len(asset_df)}"
        )


    # ========================================================
    # DISPLAY
    # ========================================================

    print("\n" + "=" * 100)

    print(
        f"🧪 SAMPLE "
        f"{i + 1}/{len(asset_df)}"
    )

    print("=" * 100)


    print("\n🔵 ORIGINAL:")
    print(source)


    print("\n🟢 SIMPLIFIED:")
    print(prediction)


    print("\n📊 METRICS")
    print("-" * 60)

    print(
        f"SARI          : "
        f"{scores['sari']:.4f}"
    )

    print(
        f"BLEU          : "
        f"{scores['bleu']:.4f}"
    )

    print(
        f"ROUGE-L       : "
        f"{scores['rougeL']:.4f}"
    )

    print(
        f"BERTScore-F1  : "
        f"{scores['bertscore_f1']:.4f}"
    )

    print(
        f"FKGL          : "
        f"{scores['fkgl']:.4f}"
    )


    print("\n⚡ PERFORMANCE")
    print("-" * 60)

    print(
        f"Inference Time: "
        f"{inference_time:.2f} sec"
    )

    if peak_vram is not None:

        print(
            f"Peak VRAM     : "
            f"{peak_vram:.3f} GB"
        )

    else:

        print(
            "Peak VRAM     : N/A"
        )


    print(
        f"\n📈 PROGRESS: "
        f"[{i + 1}/{len(asset_df)}]"
    )


    # ========================================================
    # CLEANUP
    # ========================================================

    del inputs


    if output_ids is not None:

        del output_ids


    if i % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


# ============================================================
# FINAL SAVE
# ============================================================

cot_df.to_csv(

    COT_OUTPUT,

    index=False
)


print("\n" + "=" * 100)

print(
    "🎉 QWEN CoT EVALUATION COMPLETE"
)

print("=" * 100)


print(
    f"Total samples completed: "
    f"{len(cot_df)}"
)


print(
    f"Output CSV: "
    f"{COT_OUTPUT}"
)

print("=" * 100)

🔥 QWEN2.5-3B-INSTRUCT — CHAIN-OF-THOUGHT
Model: Qwen/Qwen2.5-3B-Instruct
Samples: 2359
References per sample: 10
Maximum new tokens: 128
Checkpoint every: 25

Training: ❌ OFF
Gradients: ❌ OFF
Evaluation: ✅ ON

✅ ASSET dataset verification passed.
✅ Tokenizer verified.
✅ Qwen model verified.

📁 CHECKPOINT / RESUME SETUP
🆕 No existing checkpoint.
Starting from sample 1.

🚀 STARTING QWEN CoT EVALUATION


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Streaming output truncated to the last 5000 lines.

🧪 SAMPLE 2161/2359

🔵 ORIGINAL:
The largest populations of Mennonites are in Canada, Democratic Republic of Congo and the United States, but Mennonites can also be found in tight-knit communities in at least 51 countries on six continents or scattered amongst the populace of those countries.

🟢 SIMPLIFIED:
The largest Mennonite populations are in Canada, the Democratic Republic of Congo, and the United States, but they also live in smaller groups in over 50 countries across six continents or scattered throughout many others.

📊 METRICS
------------------------------------------------------------
SARI          : 46.9292
BLEU          : 0.5830
ROUGE-L       : 0.5274
BERTScore-F1  : 0.9355
FKGL          : 18.9629

⚡ PERFORMANCE
------------------------------------------------------------
Inference Time: 2.06 sec
Peak VRAM     : 6.778 GB

📈 PROGRESS: [2161/2359]

🧪 SAMPLE 2162/2359

🔵 ORIGINAL:
Naas is a major "Dublin Suburb" town, with m

In [10]:
# ============================================================
# 📊 QWEN2.5-3B CoT — OVERALL DESCRIPTIVE STATISTICS
# ============================================================

import pandas as pd
import numpy as np

CSV_FILE = "QWEN_COT_ASSET_2359_FINAL.csv"

print("=" * 100)
print("📂 LOADING QWEN CoT RESULTS")
print("=" * 100)

df = pd.read_csv(CSV_FILE)

print(f"\nTotal samples: {len(df)}")

# ============================================================
# VERIFY DATA
# ============================================================

assert len(df) == 2359, (
    f"Expected 2359 samples, found {len(df)}"
)

print("✅ 2359 samples verified.")

print("\nCSV COLUMNS:")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")


# ============================================================
# METRICS
# ============================================================

metrics = [
    "sari",
    "bleu",
    "rougeL",
    "bertscore_f1",
    "fkgl",
    "inference_time_sec",
    "peak_vram_gb"
]


metric_names = {
    "sari": "SARI",
    "bleu": "BLEU",
    "rougeL": "ROUGE-L",
    "bertscore_f1": "BERTScore-F1",
    "fkgl": "FKGL",
    "inference_time_sec": "Inference Time (sec)",
    "peak_vram_gb": "Peak VRAM (GB)"
}


# ============================================================
# CALCULATE STATISTICS
# ============================================================

results = []

for metric in metrics:

    values = pd.to_numeric(
        df[metric],
        errors="coerce"
    ).dropna()

    results.append({

        "Metric":
            metric_names[metric],

        "Mean":
            values.mean(),

        "Median":
            values.median(),

        "Min":
            values.min(),

        "Max":
            values.max(),

        "Std":
            values.std(),

        "Valid Samples":
            len(values),

        "Missing":
            df[metric].isna().sum()
    })


stats_df = pd.DataFrame(results)


# ============================================================
# ROUND VALUES
# ============================================================

for col in [
    "Mean",
    "Median",
    "Min",
    "Max",
    "Std"
]:

    stats_df[col] = (
        stats_df[col]
        .round(4)
    )


# ============================================================
# DISPLAY
# ============================================================

print("\n" + "=" * 100)
print("📊 QWEN2.5-3B-INSTRUCT — CoT DESCRIPTIVE STATISTICS")
print("=" * 100)

print()

print(
    stats_df.to_string(
        index=False
    )
)


# ============================================================
# SAVE STATISTICS
# ============================================================

OUTPUT_FILE = (
    "QWEN_COT_ASSET_2359_STATISTICS.csv"
)

stats_df.to_csv(
    OUTPUT_FILE,
    index=False
)


print("\n" + "=" * 100)
print("🎉 STATISTICS COMPLETE")
print("=" * 100)

print(
    f"\n📁 Saved to: {OUTPUT_FILE}"
)

📂 LOADING QWEN CoT RESULTS

Total samples: 2359
✅ 2359 samples verified.

CSV COLUMNS:
0: sample_id
1: model
2: prompting
3: original
4: generated
5: raw_generated
6: references
7: sari
8: bleu
9: rougeL
10: bertscore_f1
11: fkgl
12: inference_time_sec
13: peak_vram_gb

📊 QWEN2.5-3B-INSTRUCT — CoT DESCRIPTIVE STATISTICS

              Metric    Mean  Median     Min     Max    Std  Valid Samples  Missing
                SARI 41.5988 41.1267 15.1098 80.2756 8.6783           2359        0
                BLEU  0.5904  0.5920  0.0788  1.0000 0.2063           2359        0
             ROUGE-L  0.5572  0.5590  0.1078  0.9560 0.1282           2359        0
        BERTScore-F1  0.9483  0.9496  0.8510  0.9939 0.0165           2359        0
                FKGL  9.6774  9.7400 -1.8400 24.9577 3.6914           2359        0
Inference Time (sec)  1.2369  1.1490  0.3622  3.8653 0.4847           2359        0
      Peak VRAM (GB)  6.7749  6.7748  5.7810  6.7820 0.0205           2359        0

🎉 ST